Extracting API levels

In [1]:
# API Level Extraction from YAML Files
import os
import yaml
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\July_31\YAML_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "3.3_Project_List_API_YML_Summary.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "3.3_Project_List_API_YML_Details.csv")


# === EXTRACT ALL API LEVELS (unchanged) ===
def extract_all_api_levels(obj):
    api_levels = set()

    def recurse(o):
        if isinstance(o, dict):
            for k, v in o.items():
                key_lower = str(k).lower()

                # ✅ EXISTING: Strict match for known SDK/API-related keys
                if re.fullmatch(r'(api[-_]?level|api[-_]?versions|compile[-_]?sdk|target[-_]?sdk)', key_lower):
                    if isinstance(v, list):
                        for val in v:
                            if isinstance(val, str):
                                split_vals = re.split(r'[,\s]+', val)
                                for item in split_vals:
                                    if item.strip().isdigit():
                                        api_levels.add(item.strip())
                            elif isinstance(val, int):
                                api_levels.add(str(val))
                    elif isinstance(v, str):
                        split_vals = re.split(r'[,\s]+', v)
                        for item in split_vals:
                            if item.strip().isdigit():
                                api_levels.add(item.strip())
                    elif isinstance(v, int):
                        api_levels.add(str(v))

                # ✅ NEW: Relaxed key match for variables like ANDROID_EMULATOR_API_LEVEL
                elif re.fullmatch(r'(compile[_-]?sdk|target[_-]?sdk|api[_-]?level)', key_lower):
                    if isinstance(v, (int, str)) and str(v).isdigit():
                        api = int(v)
                        if 1 <= api <= 34:  # restrict to valid Android API range
                            api_levels.add(str(api))

                # Handle lists
                if isinstance(v, list):
                    for val in v:
                        if isinstance(val, dict):
                            recurse(val)
                        elif isinstance(val, str):
                            matches = re.findall(
                                r'(?:api[-_]?level\s*[:=]?\s*|compile[-_]?sdk\s*[:=]?\s*|target[-_]?sdk\s*[:=]?\s*|platforms;android[-_]?|android-)(\d{2,3})',
                                val,
                                flags=re.IGNORECASE
                            )
                            for match in matches:
                                api_levels.add(match)

                elif isinstance(v, dict):
                    recurse(v)

                elif isinstance(v, str):
                    recurse(v)

        elif isinstance(o, list):
            for item in o:
                recurse(item)

        elif isinstance(o, str):
            if any(kw in o.lower() for kw in ['--flavor', 'flutter build', 'gradlew', 'apk', 'assemble']):
                return

            matches = re.findall(
                r'(?:api[-_]?level\s*[:=]?\s*|compile[-_]?sdk\s*[:=]?\s*|target[-_]?sdk\s*[:=]?\s*)["\']?(\d{2,3})["\']?',
                o,
                flags=re.IGNORECASE
            )
            for match in matches:
                api_levels.add(match)

            sdkmanager_matches = re.findall(
                r'platforms;android[-_]?(\d{2,3})',
                o,
                flags=re.IGNORECASE
            )
            for match in sdkmanager_matches:
                api_levels.add(match)

    recurse(obj)
    return api_levels


# === PARSE YAML FILE (unchanged) ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            content = yaml.safe_load(raw)
            if not content:
                return {'api_levels': set(), 'error': True}
            all_api_levels = extract_all_api_levels(content)
            return {'api_levels': all_api_levels, 'error': False}
    except Exception:
        return {'api_levels': set(), 'error': True}


# === SCAN PROJECTS ===
project_results = {}
detailed_rows = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)

            # ✅ Extract full_name before first "__"
            full_name = filename.split("__")[0].lower()
            result = parse_yaml_file(file_path)

            if full_name not in project_results:
                project_results[full_name] = {
                    'api_levels': set(),
                    'errors': 0,
                    'yml_count': 0
                }
                detailed_rows[full_name] = []

            project_results[full_name]['api_levels'].update(result['api_levels'])
            project_results[full_name]['yml_count'] += 1
            if result['error']:
                project_results[full_name]['errors'] += 1

# === BUILD DETAILED ROWS ===
final_detailed_rows = []
for full_name, data in project_results.items():
    for api in data['api_levels']:
        final_detailed_rows.append({
            'filename': '',  # placeholder for below loop
            'full_name': full_name,
            'api_level': api,
            'source': 'detected',
            'yml_count': data['yml_count']
        })

# === Rebuild detailed rows with filename ===
# Note: Since each full_name may come from multiple files,
# we'll loop again to map filename properly.

detailed_output = []
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            filename = os.path.basename(file)
            full_name = filename.split("__")[0].lower()

            result = parse_yaml_file(os.path.join(root, file))
            for api in result['api_levels']:
                detailed_output.append({
                    'filename': filename,
                    'full_name': full_name,
                    'api_level': api,
                    'source': 'yml',
                })


# === EXPORT ===
df_detailed = pd.DataFrame(detailed_output)
df_detailed.to_csv(DETAILED_CSV, index=False)

summary_rows = []
for full_name, result in project_results.items():
    summary_rows.append({
        'full_name': full_name,
        'distinct_api_levels': len(result['api_levels']),
        'yaml_errors': result['errors'],
        'yaml_count': result['yml_count'],
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

print(f"\n✅ Summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Detailed CSV saved to: {DETAILED_CSV}")



✅ Summary CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_YML_Summary.csv
✅ Detailed CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_YML_Details.csv


In [2]:
import os
import pandas as pd
import re

# === CONFIG ===
BUILD_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\July_31\Build_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "3.3_Project_List_API_Gradle_Summary.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "3.3_Project_List_API_Gradle_Details.csv")

# === EXTRACT ONLY INSTRUMENTATION-RELATED API LEVELS ===
def extract_api_levels_from_gradle(content):
    api_levels = set()
    content = content.replace('\t', '    ')
    
    # ✅ Only match 'apiLevel' (from managedDevices)
    pattern = re.compile(
        r'\bapiLevel\b\s*(?:[=:])?\s*["\']?(?P<value>\d{2,3})["\']?',
        flags=re.IGNORECASE
    )

    for match in pattern.finditer(content):
        value = match.group("value")
        if value.isdigit():
            api_levels.add(value)

    return api_levels

# === SCAN GRADLE FILES ===
project_results = {}
detailed_output = []

for root, _, files in os.walk(BUILD_DIR):
    for file in files:
        if file.endswith(('.gradle', '.gradle.kts')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            full_name = filename.split("__")[0].lower()

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                api_levels = extract_api_levels_from_gradle(content)
                error = False
            except Exception:
                api_levels = set()
                error = True

            if full_name not in project_results:
                project_results[full_name] = {
                    'api_levels': set(),
                    'errors': 0,
                    'build_file_count': 0
                }

            project_results[full_name]['api_levels'].update(api_levels)
            project_results[full_name]['build_file_count'] += 1
            if error:
                project_results[full_name]['errors'] += 1

            for api in api_levels:
                detailed_output.append({
                    'filename': filename,
                    'full_name': full_name,
                    'api_level': api,
                    'source': 'gradle',
                })

# === EXPORT TO CSV ===
pd.DataFrame(detailed_output).to_csv(DETAILED_CSV, index=False)

summary_rows = [
    {
        'full_name': full_name,
        'distinct_api_levels': len(result['api_levels']),
        'build_file_errors': result['errors'],
        'build_file_count': result['build_file_count'],
    }
    for full_name, result in project_results.items()
]
pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

print(f"\n✅ Instrumentation API-level summary saved to: {SUMMARY_CSV}")
print(f"✅ Detailed API-level results saved to: {DETAILED_CSV}")



✅ Instrumentation API-level summary saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_Gradle_Summary.csv
✅ Detailed API-level results saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_Gradle_Details.csv


In [3]:
# Merge YML and Gradle API Level Data

import pandas as pd
import os

# === File Paths ===
FOLDER = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31"

# Input files
yml_summary_path = os.path.join(FOLDER, "3.3_Project_List_API_YML_Summary.csv")
yml_details_path = os.path.join(FOLDER, "3.3_Project_List_API_YML_Details.csv")
gradle_summary_path = os.path.join(FOLDER, "3.3_Project_List_API_Gradle_Summary.csv")
gradle_details_path = os.path.join(FOLDER, "3.3_Project_List_API_Gradle_Details.csv")

# Output files
merged_summary_path = os.path.join(FOLDER, "3.3_Project_List_API_Merged_Summary.csv")
merged_details_path = os.path.join(FOLDER, "3.3_Project_List_API_Merged_Details.csv")

# === Load All Files ===
df_yml_summary = pd.read_csv(yml_summary_path)
df_yml_details = pd.read_csv(yml_details_path)
df_gradle_summary = pd.read_csv(gradle_summary_path)
df_gradle_details = pd.read_csv(gradle_details_path)

# Ensure consistent casing for full_name
for df in [df_yml_summary, df_yml_details, df_gradle_summary, df_gradle_details]:
    if 'full_name' in df.columns:
        df['full_name'] = df['full_name'].str.lower()

# === Merge Details ===
df_merged_details = pd.concat([df_yml_details, df_gradle_details], ignore_index=True)
df_merged_details.to_csv(merged_details_path, index=False)

# === Merge Summaries ===
df_merged_summary = pd.merge(
    df_yml_summary,
    df_gradle_summary,
    on='full_name',
    how='outer',
    suffixes=('_yml', '_gradle')
)

df_merged_summary.to_csv(merged_summary_path, index=False)

# === Done ===
print(f"✅ Merged summary saved to: {merged_summary_path}")
print(f"✅ Merged details saved to: {merged_details_path}")


✅ Merged summary saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_Merged_Summary.csv
✅ Merged details saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_Merged_Details.csv
